# 08 - Fabric capacity benchmark and 30-day feasibility gate

Run this notebook through a benchmark pipeline over representative videos and increasing `CONCURRENT_WORKERS` levels. One activity measures one worker; the pipeline supplies concurrency. Results calculate the minimum workers required to process 200,000 video-hours in 30 days. Do not approve a capacity from a single short video or an interactive notebook run.

**After importing into Fabric:** On the configuration code cell, select **... -> Toggle parameter cell** and confirm the parameter indicator. Then attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

In [ ]:
RUN_INFERENCE = False
ENFORCE_CAPACITY_GATE = False
BENCHMARK_BATCH_ID = ""
VIDEO_URI = ""
SAMPLE_NAME = ""
EXPECTED_VIDEO_DURATION_SECONDS = 0.0
CAPACITY_SKU = "UNSET"
RUNTIME_VERSION = "UNSET"
CONCURRENT_WORKERS = 1
DATABASE = ""
TABLE_PREFIX = "people_counter"
PIPELINE = "rtdetr-osnet"
DEVICE_VARIANT = "cpu"
DEVICE = "cpu"
BATCH_SIZE = 1
SAMPLE_FPS = 3.0
DETECTION_THRESHOLD = 0.6
USE_FP16 = False
LINE = []
DETECTOR_MODEL = "r18"
CAMERA_MOTION_COMPENSATION = None
TARGET_VIDEO_HOURS = 200000.0
DEADLINE_DAYS = 30.0
UTILIZATION = 0.80
HEADROOM_FACTOR = 1.20

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from urllib.parse import urlsplit
import hashlib
import json
import math
import re
import time
import uuid

import notebookutils
from pyspark.sql import SparkSession, functions as F

from people_counter import RFDetrBotsortConfig, RTDetrOsnetConfig, run


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def parameter_bool(value: object, name: str) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"
    raise ValueError(f"{name} must be true or false")


run_inference = parameter_bool(RUN_INFERENCE, "RUN_INFERENCE")
enforce_capacity_gate = parameter_bool(ENFORCE_CAPACITY_GATE, "ENFORCE_CAPACITY_GATE")
database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")
if int(CONCURRENT_WORKERS) < 1:
    raise ValueError("CONCURRENT_WORKERS must be at least 1")
if DEVICE_VARIANT != "cpu" or DEVICE != "cpu":
    raise ValueError("Fabric-native Spark benchmark requires CPU device settings")
if USE_FP16 not in {False, "false", "False"}:
    raise ValueError("Fabric-native CPU benchmark requires USE_FP16=false")
validated_use_fp16 = False
if PIPELINE not in {"rtdetr-osnet", "rfdetr-botsort"} or DETECTOR_MODEL not in {"r18", "r50"}:
    raise ValueError("Benchmark pipeline/model selection is invalid")
if not 0 < float(UTILIZATION) <= 1 or float(HEADROOM_FACTOR) < 1:
    raise ValueError("UTILIZATION and HEADROOM_FACTOR are invalid")


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
config_value = {
    "pipeline": PIPELINE,
    "device_variant": DEVICE_VARIANT,
    "device": DEVICE,
    "batch_size": BATCH_SIZE,
    "sample_fps": SAMPLE_FPS,
    "detection_threshold": DETECTION_THRESHOLD,
    "use_fp16": validated_use_fp16,
    "line": LINE,
    "detector_model": DETECTOR_MODEL,
    "camera_motion_compensation": CAMERA_MOTION_COMPENSATION,
}
config_json = json.dumps(config_value, sort_keys=True, separators=(",", ":"))
config_sha256 = hashlib.sha256(config_json.encode("utf-8")).hexdigest()

if run_inference:
    if not VIDEO_URI or not SAMPLE_NAME or float(EXPECTED_VIDEO_DURATION_SECONDS) <= 0:
        raise ValueError("VIDEO_URI, SAMPLE_NAME, and EXPECTED_VIDEO_DURATION_SECONDS are required")
    if not BENCHMARK_BATCH_ID.strip():
        raise ValueError("BENCHMARK_BATCH_ID must identify one sustained concurrent test run")
    if not isinstance(LINE, list) or len(LINE) != 4 or any(isinstance(value, bool) or not isinstance(value, int) for value in LINE):
        raise ValueError("LINE must contain four source-video pixel coordinates")
    benchmark_id = uuid.uuid4().hex
    temporary = Path("/tmp") / f"people-counter-benchmark-{benchmark_id}"
    temporary.mkdir(mode=0o700, parents=False, exist_ok=False)
    staged = temporary / (Path(urlsplit(VIDEO_URI).path).name or "sample.video")
    benchmark_started_at = datetime.now(timezone.utc)
    started = time.perf_counter()
    succeeded = False
    error_message = None
    processing_seconds = 0.0
    end_to_end_seconds = 0.0
    speed = 0.0
    try:
        if Path(VIDEO_URI).is_file():
            staged = Path(VIDEO_URI)
        else:
            copied = notebookutils.fs.cp(VIDEO_URI, staged.as_uri())
            if copied is False or not staged.is_file():
                raise RuntimeError(f"Could not stage benchmark video: {VIDEO_URI}")
        common = {
            "video": staged,
            "device_variant": DEVICE_VARIANT,
            "device": DEVICE,
            "batch_size": int(BATCH_SIZE),
            "sample_fps": SAMPLE_FPS,
            "detection_threshold": float(DETECTION_THRESHOLD),
            "use_fp16": validated_use_fp16,
            "line": tuple(LINE) if LINE else None,
        }
        if PIPELINE == "rtdetr-osnet":
            config = RTDetrOsnetConfig(**common, detector_model=DETECTOR_MODEL)
        elif PIPELINE == "rfdetr-botsort":
            config = RFDetrBotsortConfig(
                **common,
                camera_motion_compensation=CAMERA_MOTION_COMPENSATION,
            )
        else:
            raise ValueError(f"Unsupported PIPELINE: {PIPELINE}")
        result = run(config)
        processing_seconds = result.processing_seconds
        end_to_end_seconds = time.perf_counter() - started
        speed = float(EXPECTED_VIDEO_DURATION_SECONDS) / end_to_end_seconds
        succeeded = True
    except Exception as error:
        end_to_end_seconds = time.perf_counter() - started
        error_message = f"{type(error).__name__}: {error}"[:4000]
        raise
    finally:
        row = {
            "benchmark_id": benchmark_id,
            "benchmark_batch_id": BENCHMARK_BATCH_ID,
            "benchmark_started_at": benchmark_started_at,
            "completed_at": datetime.now(timezone.utc),
            "capacity_sku": CAPACITY_SKU,
            "runtime_version": RUNTIME_VERSION,
            "sdk_version": version("people-counter"),
            "config_sha256": config_sha256,
            "sample_name": SAMPLE_NAME,
            "video_duration_seconds": float(EXPECTED_VIDEO_DURATION_SECONDS),
            "end_to_end_seconds": end_to_end_seconds,
            "overhead_seconds": max(0.0, end_to_end_seconds - processing_seconds),
            "processing_seconds": processing_seconds,
            "speed_x_realtime": speed,
            "peak_memory_mb": None,
            "concurrent_workers": int(CONCURRENT_WORKERS),
            "succeeded": succeeded,
            "error_message": error_message,
        }
        spark_session.createDataFrame(
            [row],
            spark_session.table(table("processing_benchmarks")).schema,
        ).write.format("delta").mode("append").saveAsTable(table("processing_benchmarks"))
        if staged.parent == temporary:
            staged.unlink(missing_ok=True)
        temporary.rmdir()
else:
    print("Inference disabled. Set RUN_INFERENCE=true only in the Fabric benchmark pipeline.")

In [ ]:
benchmarks = spark_session.table(table("processing_benchmarks")).where(
    (F.col("succeeded") == True)
    & (F.col("config_sha256") == config_sha256)
    & (F.col("capacity_sku") == CAPACITY_SKU)
    & (F.col("runtime_version") == RUNTIME_VERSION)
)
summary = (
    benchmarks.groupBy("benchmark_batch_id", "concurrent_workers")
    .agg(
        F.count("benchmark_id").alias("samples"),
        F.avg("speed_x_realtime").alias("average_speed_x"),
        F.expr("percentile_approx(speed_x_realtime, 0.10)").alias("p10_speed_x"),
        F.expr("percentile_approx(end_to_end_seconds, 0.95)").alias("p95_end_to_end_seconds"),
        F.sum("video_duration_seconds").alias("total_video_seconds"),
        F.min("benchmark_started_at").alias("batch_started_at"),
        F.max("completed_at").alias("batch_completed_at"),
    )
    .withColumn(
        "sustained_wall_seconds",
        F.unix_timestamp("batch_completed_at") - F.unix_timestamp("batch_started_at"),
    )
    .withColumn(
        "observed_aggregate_speed_x",
        F.col("total_video_seconds") / F.col("sustained_wall_seconds"),
    )
    .orderBy("benchmark_batch_id", "concurrent_workers")
)
display(summary)
rows = summary.collect()
if rows:
    conservative_speed = min(row.p10_speed_x for row in rows if row.p10_speed_x is not None)
    required_workers = math.ceil(
        float(TARGET_VIDEO_HOURS)
        * float(HEADROOM_FACTOR)
        / (float(DEADLINE_DAYS) * 24.0 * conservative_speed * float(UTILIZATION))
    )
    required_aggregate_speed = (
        float(TARGET_VIDEO_HOURS) * float(HEADROOM_FACTOR)
        / (float(DEADLINE_DAYS) * 24.0 * float(UTILIZATION))
    )
    sustained_rows = [
        row
        for row in rows
        if row.sustained_wall_seconds is not None and row.sustained_wall_seconds >= 6 * 3600
    ]
    best_aggregate_speed = max(
        (row.observed_aggregate_speed_x for row in sustained_rows),
        default=0.0,
    )
    outcome = {
        "target_video_hours": float(TARGET_VIDEO_HOURS),
        "deadline_days": float(DEADLINE_DAYS),
        "conservative_speed_x": conservative_speed,
        "required_workers_with_headroom": required_workers,
        "required_aggregate_speed_x": required_aggregate_speed,
        "best_six_hour_aggregate_speed_x": best_aggregate_speed,
        "capacity_gate_passed": best_aggregate_speed >= required_aggregate_speed,
    }
else:
    outcome = {"capacity_gate_passed": False, "reason": "No matching successful benchmarks"}
print(json.dumps(outcome, sort_keys=True))
if enforce_capacity_gate and not outcome["capacity_gate_passed"]:
    raise RuntimeError(f"Fabric capacity gate failed: {json.dumps(outcome, sort_keys=True)}")